In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/pakistan_used_cars_2021.csv")
print(df.shape)
df.head()
df.info()
df.tail()

(80572, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80572 entries, 0 to 80571
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Make             80572 non-null  object
 1   Model            80572 non-null  object
 2   Version          73800 non-null  object
 3   Price            80572 non-null  object
 4   Make_Year        80572 non-null  int64 
 5   CC               80572 non-null  int64 
 6   Assembly         80572 non-null  object
 7   Mileage          80572 non-null  int64 
 8   Registered City  80572 non-null  object
 9   Transmission     80572 non-null  object
dtypes: int64(3), object(7)
memory usage: 6.1+ MB


,Make,Model,Version,Price,Make_Year,CC,Assembly,Mileage,Registered City,Transmission
80567,Suzuki,Bolan,NaN,750000.0,2015,800,Local,98000,Lahore,Manual
80568,Suzuki,Alto,VXR,1480000.0,2020,660,Local,7800,Karachi,Manual
80569,Toyota,Corolla,GLi Automatic 1.3 VVTi,2700000.0,2016,1300,Local,81000,Islamabad,Automatic
80570,FAW,X-PV,Standard,600000.0,2013,1000,Local,2,Lahore,Manual
80571,Honda,Civic,VTi Prosmatec 1.8 i-VTEC,2620000.0,2013,1800,Imported,100000,Islamabad,Automatic


In [2]:
df.info()
print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())
df[["Price", "Make_Year", "CC", "Mileage"]].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80572 entries, 0 to 80571
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Make             80572 non-null  object
 1   Model            80572 non-null  object
 2   Version          73800 non-null  object
 3   Price            80572 non-null  object
 4   Make_Year        80572 non-null  int64 
 5   CC               80572 non-null  int64 
 6   Assembly         80572 non-null  object
 7   Mileage          80572 non-null  int64 
 8   Registered City  80572 non-null  object
 9   Transmission     80572 non-null  object
dtypes: int64(3), object(7)
memory usage: 6.1+ MB
Make                  0
Model                 0
Version            6772
Price                 0
Make_Year             0
CC                    0
Assembly              0
Mileage               0
Registered City       0
Transmission          0
dtype: int64
Duplicates: 1684


,Make_Year,CC,Mileage
count,80572.000000,80572.000000,80572.000000
mean,2011.724209,1404.083267,85653.660080
std,6.953399,684.458171,82241.870901
min,1990.000000,1.000000,1.000000
25%,2007.000000,1000.000000,36500.000000
50%,2013.000000,1300.000000,73000.000000
75%,2017.000000,1600.000000,110520.000000
max,2021.000000,10000.000000,999999.000000


In [3]:
print(df["Make"].nunique(), df["Model"].nunique())
print(df["Assembly"].value_counts())
print(df["Transmission"].value_counts())
print(df["Registered City"].value_counts().head(15))

66 389
Assembly
Local       54498
Imported    26074
Name: count, dtype: int64
Transmission
Automatic    40408
Manual       40164
Name: count, dtype: int64
Registered City
Lahore            27051
Karachi           18527
Islamabad         17123
Un-Registered      9282
Rawalpindi         1692
Multan             1640
Faisalabad         1210
Peshawar            864
Sialkot             457
Gujranwala          347
Bahawalpur          267
Hyderabad           190
Rahim Yar Khan      159
Sargodha            132
Quetta              100
Name: count, dtype: int64


In [4]:
price_num = pd.to_numeric(df["Price"], errors="coerce")
print("Not numeric:", price_num.isnull().sum())
print(df.loc[price_num.isnull(), "Price"].value_counts().head(20))


Not numeric: 1209
Price
Call for price    1209
Name: count, dtype: int64


In [5]:
print("CC < 500:", (df["CC"] < 500).sum(), "| CC > 6500:", (df["CC"] > 6500).sum())
print(df["CC"].value_counts().head(15))

print("Mileage < 100:", (df["Mileage"] < 100).sum(), "| Mileage >= 500000:", (df["Mileage"] >= 500000).sum())
print(df["Mileage"].value_counts().head(10))

CC < 500: 36 | CC > 6500: 5
CC
1300    16042
1000    14334
800     10591
1500     9813
1800     8976
660      6497
1600     4166
2000     2439
2700     1694
3000     1126
2800      982
2500      621
4600      488
2400      469
3400      245
Name: count, dtype: int64
Mileage < 100: 4529 | Mileage >= 500000: 522
Mileage
100000    3371
150000    1921
80000     1482
200000    1365
1         1315
70000     1284
90000     1255
50000     1183
60000     1125
85000     1037
Name: count, dtype: int64


In [6]:
price_num = pd.to_numeric(df["Price"], errors="coerce")
print("Not numeric:", price_num.isnull().sum())
print(df.loc[price_num.isnull(), "Price"].value_counts().head(10))
print(price_num.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

Not numeric: 1209
Price
Call for price    1209
Name: count, dtype: int64
count    7.936300e+04
mean     2.558124e+06
std      3.695094e+06
min      1.000000e+05
1%       3.000000e+05
5%       4.900000e+05
50%      1.725000e+06
95%      6.472500e+06
99%      1.900000e+07
max      9.500000e+07
Name: Price, dtype: float64


In [7]:
clean = df.copy()
print("Start:", len(clean))

# 1. Convert Price to numbers; bad values become NaN and are dropped
clean["Price"] = pd.to_numeric(clean["Price"], errors="coerce")
clean = clean.dropna(subset=["Price"])
print("After price conversion:", len(clean))

# 2. Remove duplicate listings
clean = clean.drop_duplicates()
print("After duplicates:", len(clean))

# 3. Missing Version is not an error, so label it
clean["Version"] = clean["Version"].fillna("Unknown")

# 4. Price limits (provisional)
clean = clean[(clean["Price"] >= 100_000) & (clean["Price"] <= 60_000_000)]
print("After price limits:", len(clean))

# 5. Engine size limits (provisional)
clean = clean[(clean["CC"] >= 600) & (clean["CC"] <= 6500)]
print("After CC limits:", len(clean))

# 6. Mileage: remove huge values, and tiny values on older cars
clean = clean[clean["Mileage"] < 500_000]
clean = clean[~((clean["Mileage"] < 100) & (clean["Make_Year"] <= 2019))]
print("After mileage limits:", len(clean))

# 7. Tidy text columns
for col in ["Make", "Model", "Version", "Registered City"]:
    clean[col] = clean[col].str.strip()

clean = clean.reset_index(drop=True)
print("Final:", clean.shape)
clean.describe()

Start: 80572
After price conversion: 79363
After duplicates: 77795
After price limits: 77783
After CC limits: 77741
After mileage limits: 75995
Final: (75995, 10)


,Price,Make_Year,CC,Mileage
count,7.599500e+04,75995.000000,75995.000000,75995.000000
mean,2.568786e+06,2011.691322,1398.953155,84171.770353
std,3.637347e+06,6.886602,672.667082,63781.698604
min,1.000000e+05,1990.000000,600.000000,1.000000
25%,9.800000e+05,2007.000000,1000.000000,40000.000000
50%,1.750000e+06,2013.000000,1300.000000,75000.000000
75%,2.875000e+06,2017.000000,1600.000000,112000.000000
max,5.950000e+07,2021.000000,6500.000000,496528.000000


In [8]:
clean.to_csv("../data/cleaned_cars.csv", index=False)